# Training EventDetector

Trains `model.model.EventDetector` on synthetic movies produced by the
`simulator` package (see `notebooks/simulator.ipynb` for how those are
built, and `simulator/pipeline.py::gen_data` for the actual generation
entry point). Covers: an on-the-fly `Dataset`/`DataLoader` wrapper around
`simulator.gen_data`, the training/validation loop, TensorBoard logging,
and checkpointing.

In [1]:
import sys
import time
from datetime import datetime
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter

project_root = Path(r"c:\Users\chem-bras5436\Documents\ml_mp_movements")
sys.path.insert(0, str(project_root))

import simulator
from model.loss import loss_fn
from model.model import EventDetector

## Dataset

`simulator.gen_data(batch_size, ...)` already generates a full batch in
one call, but wrapping it in a `torch.utils.data.Dataset` -- one sample
per `__getitem__`, via `batch_size=1` under the hood -- instead of calling
it directly lets `DataLoader` handle batching and, via `num_workers`,
parallelize sample generation across worker processes so it overlaps with
GPU compute instead of blocking it.

**Windows note:** `multiprocessing` uses `spawn` here (not `fork`), so
each worker process re-runs `import simulator` from scratch -- including
the real buffer-movie/PSF load, which takes several seconds. Passing
`persistent_workers=True` to `DataLoader` below pays that cost once per
training run instead of once per epoch.

In [2]:
class SimulatedEventDataset(Dataset):
    """On-the-fly synthetic training data for `EventDetector`.

    Each `__getitem__` call generates exactly one fresh sample via
    `simulator.gen_data(batch_size=1, ...)`. Nothing is pre-generated or
    cached: revisiting the same index later in the same epoch, or across
    epochs, produces a *different* sample -- unless `base_seed` is set, in
    which case sample `idx` is always generated with seed `(base_seed,
    idx)`, making it fully reproducible regardless of worker count or call
    order (each index maps to one fixed seed, not one position in a shared
    random stream that worker interleaving could reorder).

    `len(dataset)` is not a count of stored samples -- since the
    underlying data is generated rather than finite, it's an arbitrary
    "samples per epoch" chosen by the caller via `dataset_size`.
    """

    def __init__(
        self,
        dataset_size: int,
        validation: bool = False,
        base_seed: int | None = None,
        **gen_data_kwargs,
    ) -> None:
        """Initialize the dataset.

        Args:
            dataset_size: Number of samples per epoch (`len(dataset)`).
            validation: Forwarded to `simulator.gen_data` -- if True, draws
                background from the buffer movies held out from training
                (see `simulator.gen_data`'s `validation` argument).
            base_seed: If given, sample `idx` is always generated with seed
                `(base_seed, idx)` (reproducible). If None (default), every
                access generates fresh, non-reproducible randomness --
                appropriate for training, where more variety is better; set
                a `base_seed` for a validation set you want to compare
                across epochs/runs on the *same* samples.
            **gen_data_kwargs: Forwarded to `simulator.gen_data` (e.g.
                `length`, `mov_thumbnail_size`, `event_density_range`,
                `navg`, `ratiometric_rescale`).
        """
        self.dataset_size = dataset_size
        self.validation = validation
        self.base_seed = base_seed
        self.gen_data_kwargs = gen_data_kwargs

    def __len__(self) -> int:
        return self.dataset_size

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        """Generate one sample.

        Args:
            idx: Sample index; only used to derive a reproducible seed when
                `base_seed` is set (see class docstring).

        Returns:
            A tuple of (movie, ground_truth): `movie` has shape
            (1, T, H, W) and `ground_truth` is a dict of (C, T, H, W)
            tensors -- both without a batch dimension, since `DataLoader`
            adds that when it collates samples into a batch.
        """
        seed = None if self.base_seed is None else (self.base_seed, idx)
        movies, ground_truth = simulator.gen_data(
            batch_size=1,
            validation=self.validation,
            seed=seed,
            **self.gen_data_kwargs,
        )
        return movies[0], {key: value[0] for key, value in ground_truth.items()}

## Configuration

All the tunable knobs in one place. `feat_channels`/`min_channels` are
`EventDetector`'s own defaults -- the next section sanity-checks them
(parameter count, a real forward/backward pass) before committing to them
here. `BATCH_SIZE` starts deliberately modest: `CustomUNet` downsamples
via `MaxPool3d(kernel_size=(1, 2, 2))`, which only ever pools the spatial
(H, W) axes -- the temporal axis is never downsampled, so activation
memory scales linearly with `MOVIE_LENGTH` and `BATCH_SIZE` together,
unlike a typical 2D CNN where spatial pooling shrinks memory quickly with
depth. Scale `BATCH_SIZE` up cautiously and watch for `CUDA out of
memory`.

`BATCH_SIZE` here is only the *micro-batch* that sits on the GPU at once;
`train_one_epoch` accumulates gradients over `ACCUM_STEPS` consecutive
micro-batches before each `optimizer.step()`, so the **effective** batch
size is `BATCH_SIZE * ACCUM_STEPS` while peak activation memory stays that
of a single micro-batch. To get large-batch training dynamics on a small
GPU, raise `ACCUM_STEPS` (free, memory-wise) rather than `BATCH_SIZE`; the
loss is rescaled by `1 / ACCUM_STEPS` inside the loop so the gradient is a
mean over the effective batch and the learning rate needs no adjustment.
`ACCUM_STEPS = 1` is plain per-batch training.

In [3]:
# -- data --
MOVIE_LENGTH = 500  # frames per sampled background window (before navg cropping)
MOV_THUMBNAIL_SIZE = None  # None -> simulator's own train/val default (64 / 48 px)
NAVG = 5  # ratiometric window size; also excludes events from the first/last NAVG frames
EVENT_DENSITY_RANGE = (0.05, 2.5)  # events / um^2 / s, uniformly sampled per movie

# -- dataset / dataloader --
TRAIN_DATASET_SIZE = 512  # samples per training epoch (arbitrary -- data is generated, not finite)
VAL_DATASET_SIZE = 64  # samples per validation pass
VAL_SEED = 0  # fixed base_seed -> the same 64 validation samples every epoch, comparable across epochs
BATCH_SIZE = 1  # micro-batch actually on the GPU; sets peak memory (see note above)
ACCUM_STEPS = 8  # micro-batches accumulated per optimizer step -> effective batch = BATCH_SIZE * ACCUM_STEPS
NUM_WORKERS = 0  # DataLoader worker processes generating samples in parallel with GPU compute

# -- model --
N_CHANNELS = 1  # single-channel (grayscale) ratiometric input
FEAT_CHANNELS = 3  # EventDetector default
MIN_CHANNELS = 8  # EventDetector/CustomUNet default

# -- optimization --
LEARNING_RATE = 1e-3  # torch.optim.Adam's own default; tune from here based on the loss curve
LAMBDA_OFFSET = 1.0  # loss_fn's own default, spelled out here for visibility
LAMBDA_ORIENTATION = 0.8  # loss_fn's own default, spelled out here for visibility
NUM_EPOCHS = 50

# -- logging / checkpointing --
RUN_NAME = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = project_root / "data" / "runs" / RUN_NAME  # gitignored, see .gitignore's `data/` entry
CHECKPOINT_DIR = project_root / "data" / "checkpoints" / RUN_NAME  # also gitignored
CHECKPOINT_EVERY = 5  # epochs between periodic checkpoints, in addition to the best-val checkpoint

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

device: cuda


## Model

Sanity-checking `EventDetector`'s defaults (`feat_channels=3`,
`min_channels=8`) before training with them: parameter count, and a real
forward/backward pass at the actual training shape, to catch shape or
out-of-memory errors before waiting on any data generation.

For reference, at `feat_channels=3, min_channels=8` this is a small model
(~1.4M parameters, ~5.5MB in float32) -- almost all of it in the shared
backbone; each of the three heads is a lightweight ~500-parameter
"read-out" layer on top of the backbone's 3-channel output. That 3-channel
bottleneck is narrow for three heads with fairly different jobs
(classification, sub-pixel regression, angle regression) to share, which
is worth keeping in mind if training plateaus -- but it runs correctly and
produces a sensible loss at init (see `model.py`'s per-class bias/
zero-init), so it's a reasonable starting point rather than something to
pre-emptively change.

In [4]:
model = EventDetector(
    n_channels=N_CHANNELS, feat_channels=FEAT_CHANNELS, min_channels=MIN_CHANNELS
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"EventDetector: {n_params:,} parameters ({n_params * 4 / 1e6:.1f} MB in float32)")
for name, module in [
    ("backbone", model.backbone),
    ("heatmap_head", model.heatmap_head),
    ("offset_head", model.offset_head),
    ("orientation_head", model.orientation_head),
]:
    print(f"  {name}: {sum(p.numel() for p in module.parameters()):,}")

# Forward/backward smoke test at the real training shape (T shrinks by
# 2*NAVG once gen_data crops the ratiometric dead zone).
_smoke_t = MOVIE_LENGTH - 2 * NAVG
_smoke_hw = MOV_THUMBNAIL_SIZE or 64
_smoke_x = torch.randn(1, N_CHANNELS, _smoke_t, _smoke_hw, _smoke_hw, device=DEVICE)
_smoke_gt = {
    "heatmap": torch.zeros(1, 3, _smoke_t, _smoke_hw, _smoke_hw, device=DEVICE),
    "offset": torch.zeros(1, 2, _smoke_t, _smoke_hw, _smoke_hw, device=DEVICE),
    "orientation": torch.zeros(1, 2, _smoke_t, _smoke_hw, _smoke_hw, device=DEVICE),
}
_smoke_gt["heatmap"][0, 0, _smoke_t // 2, _smoke_hw // 2, _smoke_hw // 2] = 1.0

_t0 = time.time()
_smoke_pred = model(_smoke_x)
_smoke_loss = loss_fn(
    _smoke_pred, _smoke_gt, lambda_offset=LAMBDA_OFFSET, lambda_orientation=LAMBDA_ORIENTATION
)
_smoke_loss.backward()
model.zero_grad()
print(
    f"forward+backward OK: shape ({_smoke_t}, {_smoke_hw}, {_smoke_hw}), "
    f"batch_size=1, {time.time() - _t0:.1f}s, loss={_smoke_loss.item():.3f}"
)
del _smoke_x, _smoke_gt, _smoke_pred, _smoke_loss

EventDetector: 1,375,198 parameters (5.5 MB in float32)
  backbone: 1,373,687
  heatmap_head: 515
  offset_head: 498
  orientation_head: 498
forward+backward OK: shape (490, 64, 64), batch_size=1, 1.9s, loss=11.640


## Optimizer

Adam with its own default learning rate (1e-3) as a starting point --
`EventDetector`'s per-class heatmap bias init and offset/orientation
zero-init (see `model.py`) already give training a sensible starting
point, so this isn't compensating for a badly-initialized model; treat it
as a first guess to adjust once you can see the actual loss curve in
TensorBoard.

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Datasets and DataLoaders

Training draws background from `simulator.BUFFER_MOVIES`; validation uses
`validation=True` to draw only from `simulator.VAL_BUFFER_MOVIES` -- buffer
movies never seen during training -- with a fixed `base_seed` so the same
`VAL_DATASET_SIZE` samples are reused every epoch, making validation loss
comparable across epochs instead of being noise from a freshly-random
validation set each time.

`shuffle=False` on both: samples aren't stored anywhere to shuffle --
every index already generates independent fresh randomness (or, for
validation, a fixed-but-independent sample per index), so shuffling index
order wouldn't change anything.

In [6]:
train_dataset = SimulatedEventDataset(
    dataset_size=TRAIN_DATASET_SIZE,
    validation=False,
    length=MOVIE_LENGTH,
    mov_thumbnail_size=MOV_THUMBNAIL_SIZE,
    event_density_range=EVENT_DENSITY_RANGE,
    navg=NAVG,
)
val_dataset = SimulatedEventDataset(
    dataset_size=VAL_DATASET_SIZE,
    validation=True,
    base_seed=VAL_SEED,
    length=MOVIE_LENGTH,
    mov_thumbnail_size=MOV_THUMBNAIL_SIZE,
    event_density_range=EVENT_DENSITY_RANGE,
    navg=NAVG,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    persistent_workers=NUM_WORKERS > 0,
)

print(f"train: {len(train_dataset)} samples/epoch, {len(train_loader)} batches/epoch")
print(f"val:   {len(val_dataset)} samples/epoch, {len(val_loader)} batches/epoch")

train: 512 samples/epoch, 512 batches/epoch
val:   64 samples/epoch, 64 batches/epoch


## Logging and checkpointing

TensorBoard logs and checkpoints both go under `data/`, which is already
gitignored (see `.gitignore`), keyed by a timestamped `RUN_NAME` so
repeated runs don't overwrite each other.

In [7]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(RUN_DIR))

print(f"TensorBoard logs: {RUN_DIR}")
print(f"Checkpoints:      {CHECKPOINT_DIR}")
print(f"Launch TensorBoard with: tensorboard --logdir {RUN_DIR.parent}")

TensorBoard logs: c:\Users\chem-bras5436\Documents\ml_mp_movements\data\runs\20260909_204409
Checkpoints:      c:\Users\chem-bras5436\Documents\ml_mp_movements\data\checkpoints\20260909_204409
Launch TensorBoard with: tensorboard --logdir c:\Users\chem-bras5436\Documents\ml_mp_movements\data\runs


In [8]:
def save_checkpoint(
    path: Path,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    best_val_loss: float,
) -> None:
    """Save a resumable training checkpoint.

    Args:
        path: Destination file path.
        model: Model whose `state_dict` is saved.
        optimizer: Optimizer whose `state_dict` is saved, so Adam's moment
            estimates survive a resume instead of restarting from scratch.
        epoch: Index of the epoch just completed.
        best_val_loss: Best validation loss seen so far, saved alongside so
            resuming doesn't lose track of it.
    """
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
        },
        path,
    )


def load_checkpoint(
    path: Path,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
) -> tuple[int, float]:
    """Load a checkpoint saved by `save_checkpoint`, in place.

    Args:
        path: Checkpoint file to load.
        model: Model to load `state_dict` into.
        optimizer: If given, optimizer to load `state_dict` into too; omit
            to load only model weights (e.g. for inference-only use).

    Returns:
        A tuple of (epoch, best_val_loss) the checkpoint was saved at, so
        training can resume from `epoch + 1`.
    """
    checkpoint = torch.load(path, map_location=next(model.parameters()).device)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    return checkpoint["epoch"], checkpoint["best_val_loss"]

## Training and validation steps

In [9]:
def train_one_epoch(
    model: torch.nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    accum_steps: int = 1,
) -> float:
    """Run one training epoch with gradient accumulation.

    Gradients from `accum_steps` consecutive micro-batches are summed
    before each `optimizer.step()`, so the effective batch size is
    `loader.batch_size * accum_steps` while peak activation memory stays
    that of a single micro-batch. Each micro-batch loss is divided by
    `accum_steps` before `backward()` so the accumulated gradient is the
    *mean* over the effective batch (matching a single large-batch step),
    which is why the learning rate needs no rescaling. `accum_steps=1` is
    plain per-batch training.

    Args:
        model: Model to train (mutated in place; left in `train()` mode).
        loader: Training data loader.
        optimizer: Optimizer stepped once per `accum_steps` micro-batches
            (plus once more at epoch end if a partial group remains).
        device: Device each batch is moved to before the forward pass.
        accum_steps: Number of micro-batches to accumulate per optimizer
            step. Must be >= 1.

    Returns:
        Mean training loss over the epoch's micro-batches (the unscaled
        per-micro-batch loss, so it stays comparable across `accum_steps`).
    """
    model.train()
    total_loss = 0.0
    n_batches = 0
    pending = 0  # micro-batches accumulated since the last optimizer step

    optimizer.zero_grad()
    for movies, ground_truth in loader:
        movies = movies.to(device)
        ground_truth = {key: value.to(device) for key, value in ground_truth.items()}

        predictions = model(movies)
        loss = loss_fn(
            predictions,
            ground_truth,
            lambda_offset=LAMBDA_OFFSET,
            lambda_orientation=LAMBDA_ORIENTATION,
        )
        (loss / accum_steps).backward()
        pending += 1

        if pending == accum_steps:
            optimizer.step()
            optimizer.zero_grad()
            pending = 0

        total_loss += loss.item()
        n_batches += 1

    # Flush a trailing partial group when len(loader) isn't a multiple of
    # accum_steps, so its gradients aren't dropped. It's still scaled by
    # 1/accum_steps (not 1/pending), so this last step is slightly smaller
    # than a full one -- negligible for a large epoch, and it keeps every
    # micro-batch weighted identically.
    if pending > 0:
        optimizer.step()
        optimizer.zero_grad()

    return total_loss / n_batches


@torch.no_grad()
def evaluate(
    model: torch.nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> float:
    """Run one validation pass (no gradients, no optimizer step).

    Args:
        model: Model to evaluate (left in `eval()` mode after this call).
        loader: Validation data loader.
        device: Device each batch is moved to before the forward pass.

    Returns:
        Mean validation loss over the loader's batches.
    """
    model.eval()
    total_loss = 0.0
    n_batches = 0

    for movies, ground_truth in loader:
        movies = movies.to(device)
        ground_truth = {key: value.to(device) for key, value in ground_truth.items()}

        predictions = model(movies)
        loss = loss_fn(
            predictions,
            ground_truth,
            lambda_offset=LAMBDA_OFFSET,
            lambda_orientation=LAMBDA_ORIENTATION,
        )

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches

## Train

Each epoch: train on `TRAIN_DATASET_SIZE` fresh samples, validate on the
same `VAL_DATASET_SIZE` held-out-background samples, log both losses to
TensorBoard, and checkpoint. `latest.pt` is overwritten every epoch (for
resuming), `best.pt` only when validation loss improves, and
`epoch_N.pt` every `CHECKPOINT_EVERY` epochs (uncomment the `resume`
line below to continue an interrupted run from `latest.pt`).

In [10]:
best_val_loss = float("inf")
start_epoch = 0

# To resume an interrupted run:
# start_epoch, best_val_loss = load_checkpoint(CHECKPOINT_DIR / "latest.pt", model, optimizer)
# start_epoch += 1

print(
    f"effective batch size: {BATCH_SIZE} x {ACCUM_STEPS} = {BATCH_SIZE * ACCUM_STEPS}  "
    f"({len(train_loader) // ACCUM_STEPS} optimizer steps/epoch)"
)

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE, ACCUM_STEPS)
    val_loss = evaluate(model, val_loader, DEVICE)

    writer.add_scalar("loss/train", train_loss, epoch)
    writer.add_scalar("loss/val", val_loss, epoch)
    writer.flush()

    print(
        f"epoch {epoch + 1}/{NUM_EPOCHS}  "
        f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
        f"({time.time() - epoch_start:.1f}s)"
    )

    save_checkpoint(CHECKPOINT_DIR / "latest.pt", model, optimizer, epoch, best_val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(CHECKPOINT_DIR / "best.pt", model, optimizer, epoch, best_val_loss)
        print(f"  new best val_loss={best_val_loss:.4f} -> saved {CHECKPOINT_DIR / 'best.pt'}")

    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(CHECKPOINT_DIR / f"epoch_{epoch + 1}.pt", model, optimizer, epoch, best_val_loss)

writer.close()

effective batch size: 1 x 8 = 8  (64 optimizer steps/epoch)
epoch 1/50  train_loss=8.2654  val_loss=5.2300  (1308.0s)
  new best val_loss=5.2300 -> saved c:\Users\chem-bras5436\Documents\ml_mp_movements\data\checkpoints\20260909_204409\best.pt
epoch 2/50  train_loss=3.3574  val_loss=2.2199  (1307.1s)
  new best val_loss=2.2199 -> saved c:\Users\chem-bras5436\Documents\ml_mp_movements\data\checkpoints\20260909_204409\best.pt
epoch 3/50  train_loss=1.7341  val_loss=1.4400  (1304.9s)
  new best val_loss=1.4400 -> saved c:\Users\chem-bras5436\Documents\ml_mp_movements\data\checkpoints\20260909_204409\best.pt
epoch 4/50  train_loss=1.3168  val_loss=1.1794  (1306.5s)
  new best val_loss=1.1794 -> saved c:\Users\chem-bras5436\Documents\ml_mp_movements\data\checkpoints\20260909_204409\best.pt
epoch 5/50  train_loss=1.1506  val_loss=1.0722  (1307.5s)
  new best val_loss=1.0722 -> saved c:\Users\chem-bras5436\Documents\ml_mp_movements\data\checkpoints\20260909_204409\best.pt
epoch 6/50  train_lo

KeyboardInterrupt: 